# CNN para clasificar animales (paso a paso)

Este notebook explica el proyecto `2-CNN/`: una red neuronal convolucional
entrenada para clasificar imagenes de cinco especies (ranas, aranas,
pajaros, ballenas y monos) usando imagenes de 32x32 pixeles.

**Stack tecnologico**

- `tensorflow 2.16.1` para el modelo y el entrenamiento
- `scikit-learn 1.5.0` para los splits estratificados
- `Pillow`, `numpy`, `matplotlib` para imagenes y visualizacion
- `PyYAML` para la configuracion declarativa
- `PyQt6` y `tqdm` para GUI y barras de progreso (extensiones)

**Punto de entrada**

- Entrenar: `python train.py --config config/default.yaml`
- Predecir: `python predict.py --config config/default.yaml --model <ruta> --image <ruta>`

**Convenciones**

- Las referencias al codigo se muestran como `ruta/archivo.py:linea`.
- Los snippets son copia fiel del proyecto; no se modifica ninguna linea.
- Los notebooks `CNN.ipynb` y `CNNriesgo.ipynb` son material de referencia
  entregado por el profesor y **no se tocan**.


## 1. Arquitectura del proyecto

El proyecto sigue una organizacion por capas: configuracion, utilidades,
datos, modelos, entrenamiento e inferencia. Cada modulo tiene una sola
responsabilidad para que el flujo se pueda leer y explicar por separado.

```
2-CNN/
|-- config/
|   '-- default.yaml           # configuracion unica del proyecto
|-- train.py                   # entry point: entrenamiento
|-- predict.py                 # entry point: inferencia
|-- src/
|   |-- config.py              # YAML -> dataclasses tipadas
|   |-- utils.py               # seeds, logging, JSON
|   |-- data/
|   |   |-- datasets.py            # tf.data (alternativo)
|   |   |-- manual_datasets.py     # carga manual (usado por train.py)
|   |   |-- build_processed_dataset.py  # crudo -> procesado
|   |   '-- sample_to_test.py      # mover muestras a test/
|   |-- models/
|   |   '-- baseline_cnn.py        # arquitectura del modelo
|   |-- training/
|   |   '-- trainer.py             # compile, fit, evaluate
|   '-- inference/
|       '-- predict.py             # top-k predictions
|-- dataset/<clase>/<clase>__...jpg   # imagenes 32x32 por clase
|-- test/                            # muestras para prediccion rapida
'-- docs/                            # documentacion escrita (md)
```

**Mapa rapido de archivos clave**

| Archivo | Una linea de responsabilidad |
|---|---|
| `train.py` | pipeline completo de entrenamiento |
| `predict.py` | inferencia top-k desde archivo o carpeta |
| `config/default.yaml` | hiperparametros y rutas |
| `src/config.py` | YAML -> dataclasses frozen |
| `src/utils.py` | `set_seed`, `setup_logging`, JSON |
| `src/data/manual_datasets.py` | carga + split estratificado manual |
| `src/data/build_processed_dataset.py` | convierte dataset crudo a 32x32 JPG |
| `src/data/sample_to_test.py` | mueve N imagenes por clase a `test/` |
| `src/models/baseline_cnn.py` | arquitectura Conv2D + Dense |
| `src/training/trainer.py` | `compile_model`, `train_model`, `evaluate_model` |
| `src/inference/predict.py` | `load_image`, `predict_image` (top-k) |

**Flujo end-to-end**

```
config/default.yaml
       |
       v
+-------------------+
|  load_config()    |  (src/config.py)
+---------+---------+
          |
          v
+-------------------+
|  manual_datasets  |  -> train/val/test (numpy)
+---------+---------+
          |
          v
+-------------------+
|  baseline_cnn     |  -> tf.keras.Model
+---------+---------+
          |
          v
+-------------------+
|  trainer.compile  |  -> SGD/Adam + loss + metrics
+---------+---------+
          |
          v
+-------------------+
|  model.fit        |  -> checkpoint best_model.keras
+---------+---------+
          |
          v
+-------------------+
| model.evaluate    |  -> metrics.json
+---------+---------+
          |
          v
+-------------------+
| predict.py        |  -> top-k para una imagen o carpeta
+-------------------+
```


## 2. Paso 1: configuracion declarativa

Toda la configuracion vive en `config/default.yaml`. El codigo nunca tiene
rutas absolutas ni numeros magicos: cualquier cambio (tamano de imagen,
epochs, learning rate, etc.) se hace en el YAML.

**Contenido del YAML**

```yaml
project:
  name: cnn-animales
  seed: 42

data:
  dataset_dir: ../dataset
  image_size: [32, 32]
  channels: 3
  validation_split: 0.2
  test_split: 0.2
  batch_size: 64
  shuffle: true
  raw_source:
    path: /home/jojo/Imagenes/dataset_crudo
    class_mapping:
      aranas: aranas
      ballenas: ballenas
      pajaros: pajaros
      ranas: ranas
      simios: monos
  processing:
    jpeg_quality: 90
    allowed_extensions: [jpg, jpeg, png, bmp, tiff, webp]

training:
  epochs: 40
  learning_rate: 0.005
  optimizer: sgd
  early_stopping: false
  patience: 5

model:
  type: baseline_cnn
  dropout: 0.5

output:
  artifacts_dir: artifacts
  models_dir: models
  logs_dir: logs
```

**Carga tipada** (`src/config.py`)

El YAML se convierte en dataclasses frozen para que el resto del codigo
trabaje con objetos tipados en vez de strings. `_require()` falla con un
mensaje claro si falta una clave obligatoria.

```python
@dataclass(frozen=True)
class DataConfig:
    dataset_dir: str
    image_size: Tuple[int, int]
    channels: int
    validation_split: float
    test_split: float
    batch_size: int
    shuffle: bool


@dataclass(frozen=True)
class TrainingConfig:
    epochs: int
    learning_rate: float
    optimizer: str
    early_stopping: bool
    patience: int


@dataclass(frozen=True)
class Config:
    project: ProjectConfig
    data: DataConfig
    training: TrainingConfig
    model: ModelConfig
    output: OutputConfig


def load_config(path):
    with open(path, 'r', encoding='utf-8') as handle:
        raw = yaml.safe_load(handle)
    # ... mapea cada seccion del YAML a su dataclass ...
    return Config(project=..., data=..., training=..., model=..., output=...)
```

**Por que esta estructura?**

- Reproducibilidad: cambiando solo el YAML se entrena con otra config.
- Errores explicitos: `_require()` indica la clave faltante en el mensaje.
- Tipos claros: el resto del codigo recibe `(image_size, channels, ...)` y
  no strings sueltos.


## 3. Paso 2: utilidades compartidas

`src/utils.py` concentra las funciones que se usan desde varios modulos:
semillas, logging y serializacion JSON.

**Reproducibilidad**

```python
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
```

Con la misma semilla, el `train_test_split` estratificado, la inicializacion
de la red y el barajado del dataset producen el mismo resultado.

**Logging**

```python
def setup_logging(log_dir: str, name: str) -> logging.Logger:
    ensure_dir(log_dir)
    logger = logging.getLogger(name)
    if logger.handlers:
        return logger

    logger.setLevel(logging.INFO)
    logger.propagate = False

    timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
    log_path = os.path.join(log_dir, f'{name}-{timestamp}.log')

    formatter = logging.Formatter(
        fmt='%(asctime)s | %(levelname)s | %(name)s | %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S',
    )

    file_handler = logging.FileHandler(log_path, encoding='utf-8')
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)

    stream_handler = logging.StreamHandler(sys.stdout)
    stream_handler.setFormatter(formatter)
    logger.addHandler(stream_handler)

    return logger
```

Cada corrida genera un archivo de log con timestamp en
`config/logs/<name>-<YYYYMMDD-HHMMSS>.log`. La misma salida se duplica en
stdout, asi no hay que abrir el archivo para seguir el entrenamiento.

**Persistencia liviana**

`save_class_names` / `load_class_names` escriben y leen el archivo
`artifacts/classes.json` con la lista ordenada de clases que vio el modelo.
Esto es lo que usa `predict.py` para mapear indices a etiquetas humanas.


## 4. Paso 3: construccion del dataset procesado

`src/data/build_processed_dataset.py` toma imagenes crudas (en una carpeta
externa con una subcarpeta por especie) y las convierte en JPG de 32x32
listos para entrenar.

**Flujo de una imagen**

```
crudo/<clase_origen>/<archivo>.jpg
        |
        v
+-------------------+
| extension valida? |--no--> ignorada
+---------+---------+
          | si
          v
+-------------------+
| en class_mapping? |--no--> unmapped++
+---------+---------+
          | si
          v
+-------------------+
| load + resize a  |--fallo--> rejected++
| (32, 32, 3)      |
+---------+---------+
          |
          v
+-------------------+
| ya existe en     |--si--> skipped++
| destino?         |
+---------+---------+
          | no
          v
+-------------------+
| save_jpg(quality) |--fallo--> rejected++
+---------+---------+
          |
          v
destino/<clase_final>/<clase>__<origen>__<6d>.jpg
   written++
```

**Mapeo de clases**

El parametro `class_mapping` (definido en el YAML bajo
`data.raw_source.class_mapping`) renombra carpetas: por ejemplo `simios`
del dataset crudo se guarda como `monos` en la carpeta final.

| Carpeta origen | Carpeta final |
|---|---|
| aranas   | aranas   |
| ballenas | ballenas |
| pajaros  | pajaros  |
| ranas    | ranas    |
| simios   | monos    |

**Funciones clave**

- `iter_source_images(source_dir, allowed_extensions, logger)`: recorre el
  origen con `os.walk` y yield `(source_class, path)`.
- `load_and_resize(path, image_size)`: usa `tf.keras.utils.load_img` y
  redimensiona con `tf.image.resize(method='bilinear')`. Devuelve `None`
  si la imagen no se puede leer.
- `save_jpg(arr, dest_path, quality)`: convierte a `PIL.Image` y guarda en
  formato JPEG.
- `build_processed_dataset(...)`: orquestador. Acepta `dry_run=True` para
  contar y reportar sin escribir.
- `BuildSummary`: dataclass con `written`, `skipped_existing`, `rejected`
  por clase y totales.

**Uso tipico**

```
python -m src.data.build_processed_dataset --config config/default.yaml
```

Flags utiles:

- `--source PATH`: override de `data.raw_source.path`.
- `--output PATH`: override de `data.dataset_dir`.
- `--jpeg-quality N`: override de `data.processing.jpeg_quality` (1-100).
- `--dry-run`: cuenta y reporta sin escribir.


## 5. Paso 4: muestreo a test/

`src/data/sample_to_test.py` mueve N imagenes por clase desde `dataset/` a
`test/` para hacer pruebas rapidas con `predict.py` sin contaminar el
split de test del entrenamiento.

**Codigo relevante**

```python
def move_samples(dataset_dir, test_dir, per_class, seed):
    random.seed(seed)
    _ensure_dir(test_dir)

    results: Dict[str, int] = {}
    for class_name in sorted(os.listdir(dataset_dir)):
        class_path = os.path.join(dataset_dir, class_name)
        if not os.path.isdir(class_path):
            continue

        images = _list_images(class_path)
        if not images:
            results[class_name] = 0
            continue

        sample_count = min(per_class, len(images))
        samples = random.sample(images, sample_count)

        moved = 0
        for src in samples:
            base = os.path.basename(src)
            target_name = f'{class_name}__{base}'
            dst = os.path.join(test_dir, target_name)
            if os.path.exists(dst):
                continue
            shutil.move(src, dst)
            moved += 1
        results[class_name] = moved
    return results
```

Los archivos se renombran como `<clase>__<archivo>` para saber siempre de
que clase vienen. Por defecto mueve 5 imagenes por clase con `seed=42`.

Uso:

```
python -m src.data.sample_to_test --dataset dataset --test-dir test --per-class 5
```


## 6. Paso 5: carga manual del dataset

`src/data/manual_datasets.py` es el loader que usa `train.py`. Replica el
estilo del notebook original: `os.walk` por cada carpeta de clase, lectura
con `tf.keras.utils.load_img`, resize manual y splits con scikit-learn.

**Por que carga manual en vez de `tf.data`?**

- Replica exactamente el flujo del notebook del curso.
- Devuelve numpy arrays que se pueden inspeccionar y graficar facilmente.
- Para un dataset chico (decenas de imagenes por clase) es perfectamente
  viable cargarlo entero en memoria.

Existe una version alternativa en `src/data/datasets.py` que usa
`tf.keras.utils.image_dataset_from_directory` + cache + prefetch. Esa
version es la opcion moderna pero el entrenamiento se mantiene sobre la
manual para no desviarse del material de referencia.

**Pipeline**

```
dataset/<clase>/<archivo>.jpg
       |
       v
+-------------------+
| os.walk por clase |
+---------+---------+
          |
          v
+-------------------+
| load_img + array  |
+---------+---------+
          |
          v
+-------------------+
| resize si != size |
+---------+---------+
          |
          v
+-------------------+
| X = uint8 (N,H,W,3)|
| y = int   (N,)     |
+---------+---------+
          |
          v
+-------------------+
| train_test_split  |  -> test_split (20%)
+---------+---------+
          |
          v
+-------------------+
| /255.0 -> float32 |
| to_categorical    |
+---------+---------+
          |
          v
+-------------------+
| train_test_split  |  -> validation_split (20% del train)
+---------+---------+
          |
          v
ManualDatasetBundle(
    train_X, train_Y,
    val_X,   val_Y,
    test_X,  test_Y,
    class_names,
)
```

**Codigo clave**

```python
img_regex = re.compile(r'\.(jpg|jpeg|png|bmp|tiff)$', re.IGNORECASE)
for class_index, class_name in enumerate(class_dirs):
    for root, _, filenames in os.walk(class_path):
        for filename in filenames:
            if not img_regex.search(filename):
                continue
            file_path = os.path.join(root, filename)
            image = tf.keras.utils.load_img(file_path)
            image = tf.keras.utils.img_to_array(image)
            if image.ndim != 3:
                continue
            if (image.shape[0], image.shape[1]) != image_size:
                image = _resize_image(image, image_size)
            images.append(image)
            labels.append(class_index)

X = np.array(images, dtype=np.uint8)
y = np.array(labels)

train_X, test_X, train_Y, test_Y = train_test_split(
    X, y, test_size=test_split, random_state=seed, stratify=y
)
train_X = train_X.astype('float32') / 255.0
test_X = test_X.astype('float32') / 255.0
train_Y_one_hot = tf.keras.utils.to_categorical(train_Y, num_classes=len(class_names))
test_Y_one_hot = tf.keras.utils.to_categorical(test_Y, num_classes=len(class_names))
```

Los splits son **estratificados** (`stratify=y`), asi cada fold mantiene
la misma proporcion de clases que el dataset completo.


## 7. Paso 6: arquitectura del modelo

`src/models/baseline_cnn.py` define una red pequena y deliberadamente
cercana al notebook del curso. Una sola capa convolucional, una capa densa
y softmax al final.

**Codigo**

```python
def build_baseline_cnn(input_shape, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Conv2D(
            32, kernel_size=(3, 3), activation='linear',
            padding='same', input_shape=input_shape,
        ),
        tf.keras.layers.LeakyReLU(negative_slope=0.1),
        tf.keras.layers.MaxPooling2D(pool_size=(2, 2), padding='same'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(32, activation='linear'),
        tf.keras.layers.LeakyReLU(negative_slope=0.1),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(num_classes, activation='softmax'),
    ])
    return model
```

**Diagrama de capas**

```
Input (32, 32, 3)
   |
   v
Conv2D(32, 3x3, padding=same, linear)
   |
   v
LeakyReLU(alpha=0.1)
   |
   v
MaxPooling2D(2x2, padding=same)  -->  (16, 16, 32)
   |
   v
Dropout(0.5)
   |
   v
Flatten  --> 8192
   |
   v
Dense(32, linear)  -->  LeakyReLU(0.1)  -->  Dropout(0.5)
   |
   v
Dense(num_classes, softmax)
```

**Por que estas decisiones?**

- **Un solo bloque conv**: alineado 100% al notebook del curso. Mas facil
  de explicar y comparar con el material de referencia.
- **LeakyReLU + linear en vez de ReLU**: evita el problema de la neurona
  muerta en un dataset pequeno.
- **Dropout 0.5 en dos puntos**: regularizacion agresiva porque el
  dataset es chico y el modelo tiene muchos parametros para los datos.
- **Softmax al final**: las 5 clases son mutuamente excluyentes.


## 8. Paso 7: compilacion del modelo

`src/training/trainer.py:compile_model` toma el modelo y le inyecta el
optimizador, la loss y las metricas. Lee el `learning_rate` y el nombre
del optimizador desde la configuracion.

**Codigo**

```python
def compile_model(model, learning_rate, optimizer, logger):
    optimizer_name = optimizer.lower()
    if optimizer_name == 'sgd':
        opt = tf.keras.optimizers.SGD(
            learning_rate=learning_rate,
            decay=learning_rate / 100,
        )
    elif optimizer_name == 'adam':
        opt = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    else:
        raise ValueError(f'Unsupported optimizer: {optimizer}')

    model.compile(
        loss=tf.keras.losses.CategoricalCrossentropy(),
        optimizer=opt,
        metrics=['accuracy'],
    )
```

**Detalles importantes**

- `SGD(learning_rate, decay=lr/100)`: el decaimiento reduce el LR con cada
  paso; replica exactamente la configuracion del notebook original.
- `CategoricalCrossentropy`: se usa one-hot encoding (que es lo que produce
  `to_categorical` en el loader). Si las etiquetas fueran enteros se
  usaria `SparseCategoricalCrossentropy`.
- Metrica `accuracy`: es la metrica humana obvia para 5 clases balanceadas.

**Configuracion por defecto**

| Parametro | Valor |
|---|---|
| Optimizer | `sgd` |
| Learning rate | `0.005` |
| Epochs | `40` |
| Batch size | `64` |
| Loss | `CategoricalCrossentropy` |
| Dropout | `0.5` |


## 9. Paso 8: entrenamiento

`src/training/trainer.py:train_model` ejecuta el `model.fit` con dos
callbacks: `ModelCheckpoint` siempre activo y `EarlyStopping` opcional.

**Codigo**

```python
def train_model(model, train_ds, val_ds, epochs, early_stopping, patience,
                checkpoint_path, logger):
    callbacks = []
    if early_stopping:
        callbacks.append(
            tf.keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=patience,
                restore_best_weights=True,
            )
        )

    callbacks.append(
        tf.keras.callbacks.ModelCheckpoint(
            filepath=checkpoint_path,
            monitor='val_loss',
            save_best_only=True,
        )
    )

    history = model.fit(
        train_ds, epochs=epochs, validation_data=val_ds,
        callbacks=callbacks, verbose=1,
    )
    return history, checkpoint_path
```

**Callbacks explicados**

- `ModelCheckpoint(save_best_only=True)`: cada vez que `val_loss` mejora,
  se sobreescribe `best_model.keras`. Al final del entrenamiento el
  archivo contiene los pesos de la mejor epoca, no la ultima.
- `EarlyStopping(patience, restore_best_weights)`: si la `val_loss` no
  mejora durante N epocas seguidas, se corta el entrenamiento y se
  restauran los mejores pesos. Esta desactivado por defecto
  (`early_stopping: false`) para mantener la comparacion con el notebook.

**Por que `val_loss` y no `val_accuracy`?**

La loss es una señal mas fina: penaliza mas las predicciones muy
confiadas pero equivocadas. Es el criterio estandar cuando se quiere el
mejor modelo posible, no solo el que mas acierta.


## 10. Paso 9: pipeline completo de `train.py`

El entry point `train.py` orquesta todo: carga config, prepara datos,
construye el modelo, compila, entrena, evalua y guarda artefactos.

**Diagrama del flujo**

```
+-----------------------+
| argparse --config     |
+-----------+-----------+
            |
            v
+-----------------------+
| load_config           |
| ensure_dir (logs,     |
|   artifacts, models)  |
+-----------+-----------+
            |
            v
+-----------------------+
| setup_logging         |
| set_seed(cfg.seed)    |
+-----------+-----------+
            |
            v
+-----------------------+
| build_manual_splits   |  -> train/val/test (numpy)
+-----------+-----------+
            |
            v
+-----------------------+
| save_class_names      |  -> artifacts/classes.json
+-----------+-----------+
            |
            v
+-----------------------+
| build_baseline_cnn    |  -> model.summary() al log
+-----------+-----------+
            |
            v
+-----------------------+
| compile_model         |
+-----------+-----------+
            |
            v
+-----------------------+
| model.fit(...)        |  -> best_model.keras
+-----------+-----------+
            |
            v
+-----------------------+
| model.evaluate(test)  |  -> metrics.json
+-----------------------+
```

**Artefactos generados**

| Ruta | Contenido |
|---|---|
| `config/models/best_model.keras` | pesos de la mejor epoca (val_loss) |
| `config/artifacts/classes.json` | `{'classes': [...]}` ordenadas |
| `config/artifacts/metrics.json` | `loss: X`, `accuracy: Y` |
| `config/logs/train-<timestamp>.log` | log con hiperparametros y metricas |

Todos los paths son **relativos al archivo de config**: si se mueve el
YAML, los directorios se resuelven bien.


## 11. Paso 10: inferencia

`src/inference/predict.py` carga el modelo, recibe una imagen o una carpeta
y devuelve las `top_k` clases con sus probabilidades.

**Carga y preprocesamiento**

```python
def load_image(path, image_size, logger):
    image = tf.keras.utils.load_img(path, target_size=image_size)
    array = tf.keras.utils.img_to_array(image)
    array = array / 255.0
    return tf.expand_dims(array, axis=0)
```

La imagen se redimensiona a `image_size`, se pasa a float y se normaliza
con `/255.0` (mismo preprocesamiento que en el entrenamiento).
`expand_dims` agrega la dimension batch: el modelo espera `(B, H, W, C)`.

**Prediccion top-k**

```python
def predict_image(model, class_names, image_path, image_size, top_k=3):
    input_batch = load_image(image_path, image_size, active_logger)
    probs = model.predict(input_batch, verbose=0)[0]
    top_indices = np.argsort(probs)[::-1][:top_k]
    return [(class_names[i], float(probs[i])) for i in top_indices]
```

`np.argsort` descendente + slice da los indices de las K clases con mayor
probabilidad. Cada par `(label, prob)` se devuelve como una tupla.

**Entry point `predict.py`**

```
python predict.py \\
    --config config/default.yaml \\
    --model config/models/best_model.keras \\
    --image test/aranas__0XJES9H0FS8Z.jpg
```

Si `--image` es una carpeta, se iteran todos los `.jpg/.jpeg/.png` en orden
alfabetico y se predice uno por uno. Esto permite evaluar un lote sin
armar un script extra.

**Ejemplo de salida**

```
Prediccion para: test/aranas__0XJES9H0FS8Z.jpg
- aranas: 0.8421
- pajaros: 0.0934
- monos: 0.0412
```


## 12. Paso 11: inspeccion del dataset (celda ejecutable)

La siguiente celda no requiere TensorFlow. Solo usa `Pillow` y `numpy`
(ya listados en `requirements.txt`) para:

- Contar imagenes por clase y tamano total en disco.
- Verificar que cada imagen tenga la forma `(32, 32, 3)` y `dtype=uint8`.
- Mostrar una grilla 5xN con una muestra por clase usando matplotlib.

Los resultados ayudan a detectar problemas temprano (imagenes con otra
resolucion, archivos corruptos, clases desbalanceadas) sin tener que
lanzar el entrenamiento.


In [ ]:
import os
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name != "2-CNN":
    PROJECT_ROOT = PROJECT_ROOT / "2-CNN"
dataset_dir = PROJECT_ROOT / 'dataset'
classes = sorted(p for p in dataset_dir.iterdir() if p.is_dir())

print(f'Carpetas de clase encontradas: {[c.name for c in classes]}')
print()

EXPECTED_SHAPE = (32, 32, 3)
sample_per_class = 5

fig, axes = plt.subplots(
    len(classes), sample_per_class,
    figsize=(sample_per_class * 1.6, len(classes) * 1.6),
)
if len(classes) == 1:
    axes = np.array([axes])

totales = {}
tamanos = {}
for row, class_dir in enumerate(classes):
    archivos = sorted(p for p in class_dir.iterdir() if p.is_file())
    totales[class_dir.name] = len(archivos)
    bytes_total = sum(p.stat().st_size for p in archivos)
    tamanos[class_dir.name] = bytes_total / 1024.0  # KB
    for col in range(sample_per_class):
        ax = axes[row, col]
        if col >= len(archivos):
            ax.set_axis_off()
            continue
        path = archivos[col]
        try:
            img = Image.open(path).convert('RGB')
            arr = np.asarray(img)
            ok = arr.shape == EXPECTED_SHAPE and arr.dtype == np.uint8
        except Exception as e:
            img = None
            arr = None
            ok = False
        ax.imshow(arr if arr is not None else np.zeros((32, 32, 3), dtype=np.uint8))
        ax.set_xticks([])
        ax.set_yticks([])
        if col == 0:
            ax.set_ylabel(class_dir.name, fontsize=10)
        color = 'green' if ok else 'red'
        ax.set_title(
            f'{arr.shape if arr is not None else "ERR"}',
            fontsize=7, color=color,
        )

plt.tight_layout()
out = Path('/tmp/opencode/cnn_dataset_inspect.png')
fig.savefig(out, dpi=120)
plt.close(fig)
print(f'Grilla guardada en {out}')
print()
print(f'{"clase":<10s} {"n_imgs":>7s} {"KB":>10s}')
for nombre in sorted(totales):
    print(f'{nombre:<10s} {totales[nombre]:>7d} {tamanos[nombre]:>10.1f}')
print(f'{"TOTAL":<10s} {sum(totales.values()):>7d} {sum(tamanos.values()):>10.1f}')


## 13. Paso 12: artefactos generados

Despues de entrenar, la corrida deja cuatro artefactos en `config/`:

**`config/models/best_model.keras`**

El modelo serializado en formato nativo de Keras 3. Contiene la
arquitectura, los pesos y el estado del optimizador. Se carga con:

```python
import tensorflow as tf
model = tf.keras.models.load_model('config/models/best_model.keras')
```

**`config/artifacts/classes.json`**

Lista ordenada de clases en el mismo orden de los indices de salida del
modelo. `predict.py` lo usa para mapear probabilidades a etiquetas
humanas. Ejemplo:

```json
{
  "classes": ["aranas", "ballenas", "monos", "pajaros", "ranas"]
}
```

**`config/artifacts/metrics.json`**

Metricas finales en el split de test. Es un archivo plano con un par
`key: value` por linea:

```
loss: 0.9234
accuracy: 0.7150
```

**`config/logs/train-<timestamp>.log`**

El log completo: hiperparametros, cardinalidad de los splits, summary del
modelo, progreso de `model.fit` y metricas finales. Util para comparar
corridas sin reentrenar.

Estos archivos son los unicos productos del entrenamiento: borrarlos y
reentrenar es seguro y reproducible (mientras la seed y el YAML no
cambien).


## 14. Paso 13: decisiones de diseno

Esta seccion resume el por que de las elecciones clave del proyecto,
para poder defenderlas en clase.

**Por que 32x32 y no 21x28 (que es lo que decia el notebook original)**

El dataset del proyecto ya viene a 32x32. Pasar a 21x28 destruiria
informacion de forma innecesaria. Ademas, 32x32 es muy barato de entrenar
y mantiene la relacion de aspecto cuadrada.

**Por que un solo bloque convolucional**

La pauta del curso dice usar el CNN del notebook. Cambiar la arquitectura
romperia la comparacion. Con 1 conv + 1 dense + softmax ya se llega a un
accuracy razonable para los tamanos de dataset del curso.

**Por que SGD con decay y no Adam**

Replica exactamente la receta del notebook. SGD con decay=lr/100 es una
configuracion clasica de los 2010s que sigue funcionando bien para redes
pequenas. Adam esta disponible como alternativa (`optimizer: adam` en el
YAML) si se quiere comparar.

**Por que sin data augmentation**

El notebook no lo usa, asi que sacarlo mantiene la comparacion
directamente comparable. Para el dataset real, augmentation (rotaciones,
flips, zoom) probablemente suba el accuracy, pero es una mejora aparte.

**Por que carga manual en lugar de `tf.data`**

Replica el estilo del notebook (`os.walk` + `load_img` + arrays de
numpy). Para datasets chicos es perfectamente viable y mucho mas facil de
inspeccionar y debuggear.

**Por que `set_seed(42)`**

Reproducibilidad: con la misma semilla, los splits estratificados, la
inicializacion de la red y el barajado producen el mismo resultado.
Cualquiera puede reentrenar y obtener el mismo accuracy en test.

**Resumen rapido**

| Decision | Valor | Razon |
|---|---|---|
| Imagen | 32x32 RGB | el dataset crudo ya esta a 32x32 |
| Arquitectura | 1 conv + 1 dense + softmax | alinear con notebook |
| Activacion | LeakyReLU(0.1) + linear | evita neurona muerta |
| Optimizador | SGD(lr=0.005, decay=lr/100) | replica el notebook |
| Epochs | 40 | mejora A sobre los 20 del notebook |
| Batch size | 64 | estandar para CPU |
| Loss | CategoricalCrossentropy | one-hot |
| Dropout | 0.5 (en dos puntos) | dataset chico vs params |
| Splits | 60/20/20 estratificados | distribucion pareja por fold |
| Augmentation | no | comparacion directa con baseline |


## 15. Resumen y ejecucion

**Instalacion**

```
# El venv esta unificado en la raiz del repo (IA/.venv) y ya
# tiene todas las dependencias instaladas.
cd ../..
source .venv/bin/activate.fish
# Si hubiera que reinstalar las dependencias desde cero:
pip install -r 1-game/requirements.txt
pip install -r 2-CNN/requirements.txt
pip install -r 3-RNN/requirements.txt
```

**Preparar el dataset**

Si se parte de imagenes crudas en una carpeta externa, primero se procesan:

```
python -m src.data.build_processed_dataset --config config/default.yaml
```

Si ya se tienen imagenes 32x32 en `dataset/<clase>/`, este paso se puede
saltar.

**Entrenar**

```
python train.py --config config/default.yaml
```

El entrenamiento deja `best_model.keras`, `classes.json`, `metrics.json`
y un log con timestamp en `config/`. En una laptop sin GPU corre en
decenas de segundos.

**Predecir**

Sobre una sola imagen:

```
python predict.py --config config/default.yaml \\
    --model config/models/best_model.keras \\
    --image test/aranas__0XJES9H0FS8Z.jpg
```

Sobre una carpeta entera:

```
python predict.py --config config/default.yaml \\
    --model config/models/best_model.keras \\
    --image test/
```

**Archivos del proyecto**

- Codigo fuente: `2-CNN/src/`
- Configuracion: `2-CNN/config/default.yaml`
- Dataset: `2-CNN/dataset/<clase>/`
- Tests rapidos: `2-CNN/test/`
- Documentacion escrita: `2-CNN/docs/`
- Notebooks de referencia: `2-CNN/CNN.ipynb`, `2-CNN/CNNriesgo.ipynb`
  (no se modifican)

**Posibles mejoras**

- Agregar data augmentation (`tf.keras.layers.RandomFlip`,
  `RandomTranslation`, etc.) en el loader.
- Probar transfer learning con un backbone preentrenado (MobileNetV2,
  EfficientNet) congelado y una cabeza nueva.
- Exportar el modelo a TFLite para inferencia en dispositivos moviles.
- Reemplazar la carga manual por `tf.data` para datasets grandes.
- Implementar GUI en PyQt6 que conecte `train.py` y `predict.py` con
  un par de clicks.
